<a href="https://colab.research.google.com/github/TBGhorbanpour/Social-Awareness/blob/main/Location_based_topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Thesis/Data/Data/Data_Paper/Political_topic_labled.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
df.shape

(75455, 27)

In [18]:
df.head(1)

,Column1,ID,Date,Tweet,URL,Likes,Retweet,Replies,Source,FormalTweet,...,ner_entities_raw,ner_entities,ner_entity_count,decoded_entities,cities,city_count,sentiment_label,confidence_score,topic_id,topic_probability
0,1,1.560507e+18,2022-08-19 10:30:12,@bbrdiya تاحالا نون توموشی به کسی که اولین بار...,https://twitter.com/yourpenpalfar/status/15605...,1,0,0,iPhone,نان توموشی بارش بندر عالی میفهمند خاک هرمز ما...,...,"[{'entity_group': 'B_LOC', 'score': np.float32...","[('هرمز', 'B_LOC')]",1,"[('هرمز', 'B_LOC')]",['هرمز'],1,delighted,0.949858,3,0.344445


In [15]:
# Separate data by location
def get_city_subset(city_name):
    return df[df['cities'].str.contains(city_name, na=False)]

# Example cities
isfahan_df = get_city_subset('اصفهان')
urmia_df = get_city_subset('ارومیه')

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Function to perform LDA and return topics
def perform_lda(data, text_column='FormalTweet', n_topics=3, n_top_words=10):
    if len(data) == 0:
        return "No data available for LDA."

    # Vectorize the text data
    vectorizer = CountVectorizer(max_df=0.6, min_df=25)
    dtm = vectorizer.fit_transform(data[text_column])

    # Ensure there is a vocabulary created
    if len(vectorizer.get_feature_names_out()) == 0:
        return "Empty vocabulary; cannot perform LDA."

    # Apply LDA
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=0)
    lda.fit(dtm)

    # Get topics and words
    words = vectorizer.get_feature_names_out()
    topics = {}
    for i, topic in enumerate(lda.components_):
        topic_words = [words[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics[f'Topic {i + 1}'] = topic_words

    return topics

# Perform LDA for each location
isfahan_topics = perform_lda(isfahan_df)
urmia_topics = perform_lda(urmia_df)

In [17]:
# Function to print topics
def print_topics(location, topics):
    print(f"Topics for {location}:")
    if isinstance(topics, str):
        print(topics)
    else:
        for topic, words in topics.items():
            print(f"{topic}: {', '.join(words)}")
    print("\n")

# Print topics for each location
print_topics('Isfahan', isfahan_topics)
print_topics('Urmia', urmia_topics)
# Add more locations as needed


Topics for Isfahan:
Topic 1: اب, کارون, خوزستان, انتقال, استان, چهارمحال, یزد, خشک, حوضه, دریاچه
Topic 2: فرونشست, کشاورز, خشک, زمین, تجمع, ایران, شهر, اعتراض, بستر, پل
Topic 3: اب, سال, فرونشست, استان, شهر, خشکسالی, کشاورزی, صنایع, فولاد, زیست


Topics for Urmia:
Topic 1: خشک, زاینده, ایران, کارون, اب, اصفهان, سال, کار, خوزستان, نابود
Topic 2: اب, سال, کشاورزی, متر, سد, کاهش, مشکل, اطراف, منطقه, چاه
Topic 3: نمک, زیست, احیای, ایران, محیط, اذربایجان, احیا, نجات, دولت, غرب


